In [2]:
using BootstrapAsymptotics
using QuadGK
using Plots
using StableRNGs: StableRNG
using Statistics
using ProgressBars

In [ ]:
κ1    =  2.0 / sqrt(3* pi)
κstar = 0.200364

# γ_range = 10 .^ range(-1, stop=2, length=100)
γ_range = 0.5:0.2:3.0
Δ = 1.0
λ = 1.0

q0_list = []
q1_list = []

exp_q0_list = []
exp_q1_list = []

sample_over_teacher = 1.0

algo = Subsampling(r = 0.5)

for γ in ProgressBar(γ_range)
    Δ_add = BootstrapAsymptotics.get_additional_noise_from_kappas(κ1, κstar, γ)
    
    # 
    problem = BootstrapAsymptotics.RidgeOverparametrized(
    α       = sample_over_teacher / γ,
    Δ       = Δ + Δ_add,
    λ       = λ,
    ρ       = 1.0 - Δ_add,
    κ1      = κ1,
    κstar   = κstar,
    student_over_teacher_dim   = γ
    )

    result = BootstrapAsymptotics.state_evolution(
        problem, algo, algo
    )
   push!(q0_list, result.overlaps.Q[1, 1])
   push!(q1_list, result.overlaps.Q[1, 2])

   # 
   # do not include Δ_add in problem_exp as we generate the data with true Δ
   problem_exp = BootstrapAsymptotics.RidgeOverparametrized(
    α       = sample_over_teacher / γ,
    Δ       = Δ,
    λ       = λ,
    ρ       = 1.0,
    κ1      = κ1,
    κstar   = κstar,
    student_over_teacher_dim = γ
    )

    exp_m, exp_Q = BootstrapAsymptotics.overlaps_empirical(
        StableRNG(0), problem_exp, algo; teacher_dim=500, K=5
    )

    # extract the diagonal and offdiagonal elements of exp_Q
    diag_Q = [exp_Q[i, i] for i in 1:size(exp_Q, 1)]
    offdiag_Q = [exp_Q[i, j] for i in 1:size(exp_Q, 1), j in 1:size(exp_Q, 2) if i != j]
    push!(exp_q0_list, mean(diag_Q))
    push!(exp_q1_list, mean(offdiag_Q))

end

0.0%┣                                              ┫ 0/13 [00:02<00:-28, -2s/it]


UndefVarError: UndefVarError: `utils` not defined

In [ ]:
plot(γ_range, q0_list,label="q0", color="red")
plot!(γ_range, q1_list,label="q1", color="blue")

scatter!(γ_range, exp_q0_list , label="exp_q0", color="red")
scatter!(γ_range, exp_q1_list , label="exp_q1", color="blue")
# plot!(xscale=:log10)
# plot!(yscale=:log10)
# have the grid every power of 10 for x and y

In [ ]:
plot(γ_range, q0_list .- q1_list, label = "Ensemble variance")
plot!(xscale=:log10)